# YOLO26 Scratch Segmentation Training

This notebook trains a YOLO segmentation model from the converted dataset.

Upload either `scratch_yolo_seg/` or `scratch_yolo_seg.zip` to Colab before running the cells.

## 1. Install Ultralytics

In [ ]:
"""
    Install and import YOLO training dependencies.

    Main package:
        ultralytics: provides YOLO segmentation training, validation, and export APIs.

    Utility imports:
        Path    : filesystem paths
        shutil  : optional file/folder copying
        zipfile : dataset zip extraction
        yaml    : inspect data.yaml
"""
!python -m pip install -q -U ultralytics

from pathlib import Path
import shutil
import zipfile

import yaml
from ultralytics import YOLO

print('Ultralytics is ready')


## 2. Dataset Settings

In [ ]:
"""
    Locate the YOLO instance-segmentation dataset.

    Dataset name:
        scratch_yolo_seg

    Accepted locations:
        /content/scratch_yolo_seg/
        /content/scratch_yolo_seg.zip
        /content/data/scratch_yolo_seg/
        /content/data/scratch_yolo_seg.zip
        /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_yolo_seg/
        /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_yolo_seg.zip

    Expected YOLO layout:
        train/images, train/labels
        valid/images, valid/labels
        test/images,  test/labels
        data.yaml
"""
DATASET_NAME = 'scratch_yolo_seg'

candidate_dirs = [
    Path('/content') / DATASET_NAME,
    Path('/content/data') / DATASET_NAME,
    Path('/content/drive/MyDrive/Surface-Scratch-Detection/data') / DATASET_NAME,
    Path('/content/drive/MyDrive/Surface-Scratch-Detection') / DATASET_NAME,
]

candidate_zips = [
    Path('/content') / f'{DATASET_NAME}.zip',
    Path('/content/data') / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive/Surface-Scratch-Detection/data') / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive/Surface-Scratch-Detection') / f'{DATASET_NAME}.zip',
]

def find_dataset_root() -> Path:
    """
        Find a YOLO dataset folder or extract a YOLO dataset zip.

        Returns:
            Path to the folder containing data.yaml.
    """
    for path in candidate_dirs:
        if path.is_dir():
            return path

    for zip_path in candidate_zips:
        if not zip_path.is_file():
            continue

        extract_root = Path('/content')
        print(f'Extracting {zip_path} -> {extract_root}')
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(extract_root)

        for path in candidate_dirs:
            if path.is_dir():
                return path

    raise FileNotFoundError(
        'Dataset not found. Upload scratch_yolo_seg/ or scratch_yolo_seg.zip to /content.'
    )

DATASET_ROOT = find_dataset_root()
DATA_YAML = DATASET_ROOT / 'data.yaml'

print('Dataset root:', DATASET_ROOT)
print('data.yaml:', DATA_YAML)


## 3. Check YOLO Dataset Structure

In [ ]:
"""
    Validate the YOLO segmentation dataset before training.

    Checks:
        - train/valid/test image and label folders exist.
        - data.yaml exists.
        - every image has a matching .txt label file.
        - print how many labels are positive/non-empty.

    Note:
        Empty label files are valid for negative images, but missing label files are not.
"""
required_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

missing = [path for path in required_dirs if not path.is_dir()]
if missing:
    raise FileNotFoundError('Missing dataset folders:
' + '
'.join(str(path) for path in missing))
if not DATA_YAML.is_file():
    raise FileNotFoundError(f'data.yaml not found: {DATA_YAML}')

for split in ('train', 'valid', 'test'):
    image_dir = DATASET_ROOT / split / 'images'
    label_dir = DATASET_ROOT / split / 'labels'
    images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}])
    labels = sorted(label_dir.glob('*.txt'))
    missing_labels = [p.name for p in images if not (label_dir / f'{p.stem}.txt').is_file()]
    positive_labels = [p for p in labels if p.stat().st_size > 0]
    print(
        f'{split}: images={len(images)} labels={len(labels)} '
        f'positive_labels={len(positive_labels)} missing_labels={len(missing_labels)}'
    )
    if missing_labels:
        raise RuntimeError(f'{split} has missing labels. Example: {missing_labels[:5]}')

with DATA_YAML.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

print('
data.yaml:')
print(yaml.safe_dump(data_config, sort_keys=False))


## 4. Train YOLO26 Segmentation

In [ ]:
"""
    Train YOLO26 segmentation for scratch instance segmentation.

    Model:
        MODEL = yolo26n-seg.pt
            n model is the lighter/faster baseline.

    Training settings:
        EPOCHS   : 100
        IMGSZ    : 512, matches current YOLO patch dataset size.
        BATCH    : 16, adjust down if Colab runs out of VRAM.
        PATIENCE : 30 epochs without improvement before early stopping.
        WORKERS  : 2 dataloader workers for Colab stability.

    Output:
        /content/yolo_scratch_runs/scratch_yolo26n_seg/weights/best.pt
        /content/yolo_scratch_runs/scratch_yolo26n_seg/weights/last.pt
"""
MODEL = 'yolo26n-seg.pt'
PROJECT = '/content/yolo_scratch_runs'
RUN_NAME = 'scratch_yolo26n_seg'

EPOCHS = 100
IMGSZ = 512
BATCH = 16
PATIENCE = 30
WORKERS = 2

"""Load pretrained YOLO segmentation checkpoint."""
model = YOLO(MODEL)

"""Start Ultralytics segmentation training."""
results = model.train(
    data=str(DATA_YAML),
    task='segment',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    plots=True,
    save=True,
    device=0,
)

"""Resolve important run artifact paths for later cells."""
RUN_DIR = Path(PROJECT) / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'

print('Run dir:', RUN_DIR)
print('Best checkpoint:', BEST_PT, BEST_PT.exists())
print('Last checkpoint:', LAST_PT, LAST_PT.exists())


## 5. Validate Best Checkpoint

In [ ]:
"""
    Validate the best YOLO checkpoint on the test split.

    Input:
        BEST_PT from the training run.

    Metrics:
        Ultralytics prints segmentation metrics such as precision, recall,
        mAP, and mask metrics for the configured test split.

    Output:
        Validation plots/metrics are saved under the YOLO run directory.
"""
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML),
    task='segment',
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    plots=True,
    device=0,
)

print(metrics)


## 6. Download Checkpoints

In [ ]:
"""
    Download YOLO checkpoints from Colab.

    Files:
        best.pt : best validation checkpoint, usually used for inference/export.
        last.pt : last training checkpoint, useful for resume/debugging.
"""
from google.colab import files

if BEST_PT.is_file():
    files.download(str(BEST_PT))
if LAST_PT.is_file():
    files.download(str(LAST_PT))
